In [20]:
import pandas as pd
import re

def parse_can_log(file_path):
    """Parse CAN log file and return a pandas DataFrame"""
    data = []
    
    with open(file_path, 'r') as f:
        for line in f:
            # Extract fields using regex
            match = re.search(r'Timestamp:\s([\d.]+)\s+ID:\s(\w+)\s+(\w+)\s+DLC:\s(\d+)\s+(.*)', line)
            if match:
                timestamp, can_id, flags, dlc, hex_data = match.groups()
                data.append({
                    'Timestamp': float(timestamp),
                    'ID': can_id,
                    'Flags': flags,
                    'DLC': int(dlc),
                    'Data': hex_data.strip()
                })
    
    df = pd.DataFrame(data)
    return df

# Usage
df = parse_can_log(r'C:\Users\nb0801\Documents\GitHub\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks\datasets\HCRL original\normal_run_data\normal_run_data.txt')
df["Attack"] = "R"
print(df)

           Timestamp    ID Flags  DLC                     Data Attack
0       1.479121e+09  0350   000    8  05 28 84 66 6d 00 00 a2      R
1       1.479121e+09  02c0   000    8  14 00 00 00 00 00 00 00      R
2       1.479121e+09  0430   000    8  00 00 00 00 00 00 00 00      R
3       1.479121e+09  04b1   000    8  00 00 00 00 00 00 00 00      R
4       1.479121e+09  01f1   000    8  00 00 00 00 00 00 00 00      R
...              ...   ...   ...  ...                      ...    ...
988866  1.479122e+09  02b0   000    5           ac 05 0c 07 7f      R
988867  1.479122e+09  0316   000    8  05 38 10 0c 38 28 01 7a      R
988868  1.479122e+09  018f   000    8  fe 31 00 00 00 4b 00 00      R
988869  1.479122e+09  0260   000    8  32 38 39 30 ff 93 59 1c      R
988870  1.479122e+09  02a0   000    8  20 00 75 1d 01 04 dd 00      R

[988871 rows x 6 columns]


In [21]:
def parse_can_csv(file_path):
    df_can = pd.read_csv(file_path, skipinitialspace=True)

    if 'ID dlc' in df_can.columns:
        df_can[['ID', 'DLC']] = df_can['ID dlc'].astype(str).str.split(r'\s+', n=1, expand=True)
        df_can = df_can.drop(columns=['ID dlc'])
    elif set(['Timestamp','ID','DLC','data1','data2','data3','data4','data5','data6','data7','data8','attack/nonattack']).issubset(df_can.columns):
        df_can = df_can.rename(columns={'attack/nonattack': 'Attack'})
    else:
        names = ['Timestamp','ID','DLC','data1','data2','data3','data4','data5','data6','data7','data8','Attack']
        df_can = pd.read_csv(file_path, header=None, names=names, skipinitialspace=True)

    unnamed = [c for c in df_can.columns if isinstance(c, str) and c.startswith('Unnamed')]
    if 'Attack' not in df_can.columns and unnamed:
        df_can = df_can.rename(columns={unnamed[-1]: 'Attack'})

    data_cols = sorted(
        [c for c in df_can.columns if isinstance(c, str) and c.lower().startswith('data')],
        key=lambda x: int(re.search(r'\d+', x).group()) if re.search(r'\d+', x) else 0
    )

    def build_data(row):
        try:
            dlc = int(row['DLC'])
        except Exception:
            dlc = 0

        values = []
        for col in data_cols[:min(dlc, len(data_cols))]:
            val = row.get(col)
            if pd.isna(val):
                continue
            s = str(val).strip()
            if s.lower() in ['', 'nan']:
                continue
            values.append(s)

        return ' '.join(values).strip()

    def infer_attack(row):
        attack = row.get('Attack')
        if isinstance(attack, str) and attack.strip():
            return attack

        try:
            dlc = int(row['DLC'])
        except Exception:
            dlc = 0

        for col in data_cols[min(dlc, len(data_cols)):]:
            val = row.get(col)
            if pd.isna(val):
                continue
            s = str(val).strip()
            if s.lower() in ['', 'nan']:
                continue
            if not re.fullmatch(r'[0-9A-Fa-f]{2}', s):
                return s

        return pd.NA

    if data_cols:
        df_can['Data'] = df_can.apply(build_data, axis=1)
        if 'Attack' not in df_can.columns:
            df_can['Attack'] = pd.NA
        df_can['Attack'] = df_can.apply(infer_attack, axis=1).combine_first(df_can['Attack'])
    else:
        df_can['Data'] = df_can.get('Data', '').astype(str).str.strip()
        if 'Attack' not in df_can.columns:
            df_can['Attack'] = pd.NA

    df_can['Timestamp'] = pd.to_numeric(df_can['Timestamp'], errors='coerce')
    df_can['DLC'] = pd.to_numeric(df_can['DLC'], errors='coerce').astype('Int64')

    return df_can[['Timestamp','ID','DLC','Data','Attack']]

In [22]:
import numpy as np

def convert_to_array(df, attack_label='T'):
    ids = df['ID'].to_numpy(dtype=object)
    data = df['Data'].to_numpy(dtype=object)
    attack = df['Attack'].to_numpy(dtype=object)

    samples = np.column_stack((ids, data))
    labels = np.where(attack == attack_label, attack_label, 'R').astype(object)

    return samples, labels


In [23]:
id_data0, attack_labels0 = convert_to_array(df, attack_label='D')


In [24]:
df_DoS = parse_can_csv(r'C:\Users\nb0801\Documents\GitHub\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks\datasets\HCRL original\DoS_dataset.csv')
print(df_DoS.head())
id_data1, attack_labels1 = convert_to_array(df_DoS, attack_label='D')

del df_DoS

      Timestamp    ID  DLC                     Data Attack
0  1.478198e+09  0316    8  05 21 68 09 21 21 00 6f      R
1  1.478198e+09  018f    8  fe 5b 00 00 00 3c 00 00      R
2  1.478198e+09  0260    8  19 21 22 30 08 8e 6d 3a      R
3  1.478198e+09  02a0    8  64 00 9a 1d 97 02 bd 00      R
4  1.478198e+09  0329    8  40 bb 7f 14 11 20 00 14      R


In [25]:
df_Fuzzy = parse_can_csv(r'C:\Users\nb0801\Documents\GitHub\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks\datasets\HCRL original\Fuzzy_dataset.csv')
print(df_Fuzzy.head())
id_data2, attack_labels2 = convert_to_array(df_Fuzzy, attack_label='F')

del df_Fuzzy

      Timestamp    ID  DLC                     Data Attack
0  1.478196e+09  0545    8  d8 00 00 8a 00 00 00 00      R
1  1.478196e+09  02b0    5           ff 7f 00 05 49      R
2  1.478196e+09  0002    8  00 00 00 00 00 01 07 15      R
3  1.478196e+09  0153    8  00 21 10 ff 00 ff 00 00      R
4  1.478196e+09  0130    8  19 80 00 ff fe 7f 07 60      R


In [26]:
df_gear = parse_can_csv(r'C:\Users\nb0801\Documents\GitHub\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks\datasets\HCRL original\gear_dataset.csv')
print(df_gear.head())
id_data3, attack_labels3 = convert_to_array(df_gear, attack_label='S')

del df_gear

      Timestamp    ID  DLC                     Data Attack
0  1.478193e+09  0140    8  00 00 00 00 10 29 2a 24      R
1  1.478193e+09  02c0    8  15 00 00 00 00 00 00 00      R
2  1.478193e+09  0350    8  05 20 44 68 77 00 00 7e      R
3  1.478193e+09  0370    8  00 20 00 00 00 00 00 00      R
4  1.478193e+09  043f    8  10 40 60 ff 78 c4 08 00      R


In [27]:
df_RPM = parse_can_csv(r'C:\Users\nb0801\Documents\GitHub\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks\datasets\HCRL original\RPM_dataset.csv')
print(df_RPM.head())
id_data4, attack_labels4 = convert_to_array(df_RPM, attack_label='S')

del df_RPM

      Timestamp    ID  DLC                     Data Attack
0  1.478191e+09  0316    8  05 22 68 09 22 20 00 75      R
1  1.478191e+09  018f    8  fe 3b 00 00 00 3c 00 00      R
2  1.478191e+09  0260    8  19 22 22 30 ff 8f 6e 3f      R
3  1.478191e+09  02a0    8  60 00 83 1d 96 02 bd 00      R
4  1.478191e+09  0329    8  dc b8 7e 14 11 20 00 14      R


In [28]:
len(id_data0)

988871

In [29]:
# merge windowed datasets into the single variables used by downstream cells
id_data = []
id_data.extend(id_data0[int(len(id_data0)*0.2):])
id_data.extend(id_data1[int(len(id_data1)*0.2):])
id_data.extend(id_data2[int(len(id_data2)*0.2):])
id_data.extend(id_data3[int(len(id_data3)*0.2):])
id_data.extend(id_data4[int(len(id_data4)*0.2):])

id_data_test = []
id_data_test.extend(id_data0[:int(len(id_data0)*0.2)])
id_data_test.extend(id_data1[:int(len(id_data1)*0.2)])
id_data_test.extend(id_data2[:int(len(id_data2)*0.2)])
id_data_test.extend(id_data3[:int(len(id_data3)*0.2)])
id_data_test.extend(id_data4[:int(len(id_data4)*0.2)])

attack_labels = np.concatenate([
    attack_labels0[int(len(id_data0)*0.2):],
    attack_labels1[int(len(id_data1)*0.2):],
    attack_labels2[int(len(id_data2)*0.2):],
    attack_labels3[int(len(id_data3)*0.2):],
    attack_labels4[int(len(id_data4)*0.2):]
])

attack_labels_test = np.concatenate([
    attack_labels0[:int(len(id_data0)*0.2)],
    attack_labels1[:int(len(id_data1)*0.2)],
    attack_labels2[:int(len(id_data2)*0.2)],
    attack_labels3[:int(len(id_data3)*0.2)],
    attack_labels4[:int(len(id_data4)*0.2)]
])

print("merged windows:", len(id_data), "merged labels:", attack_labels.shape)

merged windows: 14046678 merged labels: (14046678,)


In [30]:
print(len(id_data), id_data[:10])

14046678 [array(['0140', '00 00 00 00 0e 25 21 8d'], dtype=object), array(['0316', '05 27 7c 0b 27 20 0b 7c'], dtype=object), array(['018f', 'fe 2d 00 00 00 4d 00 00'], dtype=object), array(['0260', '20 27 25 30 ff 93 5d 0b'], dtype=object), array(['02a0', '00 00 6b 1d 01 04 dd 00'], dtype=object), array(['0329', '40 b9 7f 14 11 20 00 14'], dtype=object), array(['0350', '05 28 c4 66 6a 00 00 e5'], dtype=object), array(['0545', 'd8 00 00 8d 00 00 00 00'], dtype=object), array(['04f0', '00 18 00 80 00 52 03 14'], dtype=object), array(['043f', '03 45 60 ff 5e 39 09 00'], dtype=object)]


In [31]:
def _hex_byte_list_from_row(row):
    msg_id, data = row
    msg_id = str(msg_id).strip()
    if len(msg_id) % 2 != 0:
        msg_id = '0' + msg_id
    id_bytes = [msg_id[i:i+2] for i in range(0, len(msg_id), 2)]
    data_bytes = [b for b in str(data).split() if b]
    return [int(b, 16) for b in id_bytes + data_bytes]

def convert_windowed_id_data_to_ints(windowed_data, pad_value=0):
    int_windows = []
    for window in windowed_data:
        rows = [_hex_byte_list_from_row(row) for row in window]
        max_len = max(len(r) for r in rows)
        padded_rows = [r + [pad_value] * (max_len - len(r)) for r in rows]
        int_windows.append(np.array(padded_rows, dtype=np.uint8))
    return int_windows

int_id_data = convert_windowed_id_data_to_ints([id_data])[0]
int_id_data_test = convert_windowed_id_data_to_ints([id_data_test])[0]

print(int_id_data[0].shape)
print(int_id_data[0][:2])

(10,)
[ 1 64]


In [32]:
int_id_data

array([[  1,  64,   0, ...,  37,  33, 141],
       [  3,  22,   5, ...,  32,  11, 124],
       [  1, 143, 254, ...,  77,   0,   0],
       ...,
       [  2, 160,  36, ...,   2, 189,   0],
       [  3,  41, 220, ...,  32,   0,  20],
       [  5,  69, 216, ...,   0,   0,   0]],
      shape=(14046678, 10), dtype=uint8)

In [33]:
attack_onehot = np.zeros((len(attack_labels), 4), dtype=np.uint8)
attack_onehot[attack_labels == 'R', 0] = 1
attack_onehot[attack_labels == 'D', 1] = 1
attack_onehot[attack_labels == 'F', 2] = 1
attack_onehot[attack_labels == 'S', 3] = 1

print(attack_onehot[:10])

attack_onehot_test = np.zeros((len(attack_labels_test), 4), dtype=np.uint8)
attack_onehot_test[attack_labels_test == 'R', 0] = 1
attack_onehot_test[attack_labels_test == 'D', 1] = 1
attack_onehot_test[attack_labels_test == 'F', 2] = 1
attack_onehot_test[attack_labels_test == 'S', 3] = 1

print(attack_onehot_test[:10])

[[1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]]
[[1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]]


In [34]:
len(int_id_data)

14046678

In [35]:
len(attack_onehot)

14046678

In [37]:
int_id_data, attack_onehot = np.array(int_id_data), np.array(attack_onehot)
int_id_data_test, attack_onehot_test = np.array(int_id_data_test), np.array(attack_onehot_test)

In [39]:
import numpy as np

# Save training data
np.save("saved_data/int_id_data.npy", int_id_data)
np.save("saved_data/attack_onehot.npy", attack_onehot)
np.save("saved_data/attack_labels.npy", attack_labels)

# Save test data
np.save("saved_data/int_id_data_test.npy", int_id_data_test)
np.save("saved_data/attack_onehot_test.npy", attack_onehot_test)
np.save("saved_data/attack_labels_test.npy", attack_labels_test)

print("Arrays saved successfully.")

Arrays saved successfully.
